# STEP 2: Load into DuckDB and profile the data

In [ ]:
## 1. setup
import os
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
print("Working in:", Path.cwd())

con = duckdb.connect("kkbox.duckdb")
con.sql("SET memory_limit='6GB'")          # about half your RAM is a safe choice
con.sql("SET temp_directory='data/tmp'")   # where DuckDB spills if memory runs out

def run_sql_file(path):
    """Run every statement in a .sql file."""
    for stmt in Path(path).read_text(encoding="utf-8").split(";"):
        if stmt.strip():
            con.sql(stmt)
    print("Done:", path)

Working in: d:\0statistic project\subscription-lifecycle-churn-analysis


In [ ]:
## 2. run the load script.
run_sql_file("sql/01_load.sql")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done: sql/01_load.sql


In [ ]:
## 3a. quick overview of every column.
con.sql("SUMMARIZE raw_tx").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,msno,VARCHAR,+++FOrTS7ab3tIgIh8eWwX4FqRv8w/FoiOuyXsFvphY=,zzzyOgMk9MljCerbCCYrVtvu85aSCiy7yCMjAEgNYMs=,2485491,None,None,None,None,None,22978755,0.0
1,payment_method_id,BIGINT,1,41,41,38.86990927054142,3.6241488260974988,38,41,41,22978755,0.0
2,payment_plan_days,BIGINT,0,450,44,33.49868532912249,39.85553421111872,30,30,30,22978755,0.0
3,plan_list_price,BIGINT,0,2000,53,148.7220067840925,170.45128537663734,99,149,149,22978755,0.0
4,actual_amount_paid,BIGINT,0,2000,63,150.6641339358899,171.46141035636902,99,149,149,22978755,0.0
5,is_auto_renew,BIGINT,0,1,2,0.8478146444400491,0.3592004715030488,1,1,1,22978755,0.0
6,transaction_date,BIGINT,20150101,20170331,1120,20158275.175349884,6627.848455211179,20150922,20160412,20161022,22978755,0.0
7,membership_expire_date,BIGINT,19700101,20361015,4412,20159401.71942257,8128.208384765982,20151073,20160538,20161131,22978755,0.0
8,is_cancel,BIGINT,0,1,2,0.038817768847790056,0.1931604289009137,0,0,0,22978755,0.0


In [8]:
## 3b. quick overview of every column.
con.sql("SUMMARIZE raw_members").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,msno,VARCHAR,+++4vcS9aMH7KWdfh5git6nA5fC5jjisd5H/NcM++WM=,zzzyOgMk9MljCerbCCYrVtvu85aSCiy7yCMjAEgNYMs=,9269002,None,None,None,None,None,6769473,0.00
1,city,BIGINT,1,22,21,3.847357689439045,5.478359063260457,1,1,4,6769473,0.00
2,bd,BIGINT,-7168,2016,330,9.795794295951842,17.92589971901099,0,0,21,6769473,0.00
3,gender,VARCHAR,female,male,2,None,None,None,None,None,6769473,65.43
4,registered_via,BIGINT,-1,19,21,5.253068739619761,2.361398314806479,4,4,7,6769473,0.00
5,registration_init_time,BIGINT,20040326,20170429,6568,20145175.906336136,23186.007612107747,20140436,20150997,20160587,6769473,0.00


In [7]:
## 3c. quick overview of every column.
con.sql("SUMMARIZE raw_logs").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,msno,VARCHAR,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,zzzueVTwIa5TjXnG2c77bohCVkuksqLkd5mQTP0wTwQ=,1018965,None,None,None,None,None,18396362,0.0
1,date,BIGINT,20170301,20170331,29,20170316.085612632,8.91672003472791,20170308,20170316,20170324,18396362,0.0
2,num_25,BIGINT,0,5639,992,6.191400941120858,13.428266336534858,0,2,7,18396362,0.0
3,num_50,BIGINT,0,912,365,1.5087888572751504,3.9085385578204668,0,1,2,18396362,0.0
4,num_75,BIGINT,0,508,209,0.9413759089976594,1.9248401148456706,0,0,1,18396362,0.0
5,num_985,BIGINT,0,1561,368,1.0799049290289027,3.5184090468639893,0,0,1,18396362,0.0
6,num_100,BIGINT,0,41107,1357,30.28246090178047,42.036411795200124,7,17,37,18396362,0.0
7,num_unq,BIGINT,1,4925,1016,29.03614551616238,32.19865987108652,8,18,38,18396362,0.0
8,total_secs,DOUBLE,0.001,9194058.522,10864794,7904.81355635642,10136.32301516662,1961.6762295057645,4588.6262591669365,9871.555004414655,18396362,0.0


In [9]:
## 4. size and date coverage
con.sql("""
    SELECT COUNT(*)              AS n_rows,
           COUNT(DISTINCT msno)  AS n_users,
           MIN(transaction_date) AS first_date,
           MAX(transaction_date) AS last_date
    FROM raw_tx
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_users,first_date,last_date
0,22978755,2426143,20150101,20170331


In [ ]:
## 5 plan mix
con.sql("""
    SELECT payment_plan_days, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM raw_tx
    GROUP BY 1 
    ORDER BY n DESC 
    LIMIT 10
""").df()

,payment_plan_days,n,pct
0,30,20174288,87.8
1,0,872342,3.8
2,31,766612,3.3
3,7,589807,2.6
4,410,162236,0.7
5,195,138802,0.6
6,180,76172,0.3
7,10,38632,0.2
8,90,31440,0.1
9,100,28252,0.1


In [11]:
## 6. auto-renew, cancellations, discounts, free periods
con.sql("""
    SELECT ROUND(AVG(is_auto_renew), 3)                            AS auto_renew_share,
           ROUND(AVG(is_cancel), 3)                                AS cancel_share,
           ROUND(AVG((actual_amount_paid < plan_list_price)::INT), 3) AS discounted_share,
           ROUND(AVG((actual_amount_paid = 0)::INT), 3)            AS free_share
    FROM raw_tx
""").df()

,auto_renew_share,cancel_share,discounted_share,free_share
0,0.848,0.039,0.038,0.053


In [12]:
## 7. churn rate in the labels
con.sql("""
    SELECT is_churn, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM labels GROUP BY 1
""").df()

,is_churn,n,pct
0,0,883630,91.0
1,1,87330,9.0


In [13]:
## 8. data quality in members
con.sql("""
    SELECT COUNT(*)                                     AS n_members,
           COUNT(*) FILTER (WHERE bd <= 0 OR bd > 90)   AS bad_age,
           COUNT(*) FILTER (WHERE gender IS NULL)       AS missing_gender
    FROM raw_members
""").df()

,n_members,bad_age,missing_gender
0,6769473,4548209,4429505


In [14]:
## 9. how many labelled users have transactions and member data
con.sql("""
    SELECT COUNT(*)                                                   AS labelled_users,
           COUNT(*) FILTER (WHERE msno IN (SELECT msno FROM raw_tx))      AS with_transactions,
           COUNT(*) FILTER (WHERE msno IN (SELECT msno FROM raw_members)) AS with_member_info
    FROM labels
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,labelled_users,with_transactions,with_member_info
0,970960,970960,860967


## Summary of findings

**Coverage**
- Transactions: 22,978,755 rows for 2,426,143 users (≈ 9.5 transactions per user), from 2015-01-01 to 2017-03-31.
- Members: 6,769,473 users. Labels (`train_v2`): 970,960 users.

**Subscription model**
- 30-day plans dominate (87.8% of transactions). Longer plans (90–410 days) are rare (< 2% combined).
- 7-day plans (2.6%) and free transactions (5.3%) suggest trials or promotions → flag them in Step 3.
- 3.8% of transactions have `payment_plan_days = 0`, close to the cancellation share (3.9%). Hypothesis: these are mostly cancellation records → check in Step 3.
- Auto-renew is the norm: 84.8% of transactions. Discounts are rare (3.8%).

**Churn**
- Churn rate among labelled users is 9.0% (87,330 of 970,960). The classes are imbalanced, so accuracy alone is a misleading metric; use PR-AUC and lift in Step 8.

**Data quality**
- Age (`bd`) is invalid (≤ 0 or > 90) for 67% of members, and gender is missing for 65%. Demographics are therefore weak variables; keep "Unknown" as its own category rather than dropping users.
- All labelled users have transactions, but 11% (109,993) have no member record. Their demographics become "Unknown".

**Implications for the next steps**
- Subscription behaviour (plan, auto-renew, payment, engagement) will matter far more than demographics.
- Step 3 cleaning: convert integer dates, flag trials/free periods, verify what 0-day plans are, and recode invalid ages.

In [15]:
  con.close()